# Step 20b — The census along a validated canonical coordinate

**Why.** Step 20 showed that the raw log-odds is about 27% nonlinear relative to its own calibrated log-ratio. Moving to a recalibrated coordinate shifted $\sigma_{\rm bif}$ by 1.5× and $h$ by $r=0.68$. But some spline calibrations were non-monotone, which makes the recalibrated coordinate an invalid reparameterisation. This notebook does the calibration properly.

**Monotone calibration.** $u(z)=\beta_0+e^{\theta_0}z+\sum_k e^{\theta_k}\,s\,\mathrm{softplus}((z-t_k)/s)-\sum_k e^{\phi_k}\,s\,\mathrm{softplus}(-(z-t_k)/s)$ combines convex and concave ramps with positive weights. It is smooth and strictly increasing by construction, and it shrinks towards an affine map. It is fitted by maximum likelihood on class labels (logistic loss), so $u$ estimates the Bayes log-ratio *given the classifier's features*.

**Cross-fitting.** Every test image is scored by a calibration fitted on the other half of the images.

**Checks.** Calibration quality is checked with the tilting identity: the binned empirical log-ratio against $u$ should have slope 1 and be linear. Along the calibrated coordinate we then redo:
- the census, with 300 bootstrap resamples;
- the symmetric null (the 30-class classifier's flip/swap pairs, recomputed here);
- the ResNet–VGG agreement (both should move towards the same answer if calibration helps);
- the hierarchy and tighter-class permutation tests, and the Gaussian baseline.

**Inputs.** Mount the committed **Step 20 output**, which contains `step20_logits.npz`, `noisy_resnet30.pt` and `noisy_resnet_seed0.pt`. GPU on (about 25 minutes, for the null logits and the Gaussian baseline); the rest is CPU. About 1.5 hours in total.

In [ ]:
import os; os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import glob, math, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
from scipy.stats import spearmanr
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); plt.rcParams['figure.dpi'] = 120
rng = np.random.default_rng(0); torch.manual_seed(0)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = {0, 1, 8, 9}; PAIRS = list(combinations(range(10), 2))
ds = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True); dst = datasets.CIFAR10('/kaggle/working/cifar', train=False, download=True)
XTR = torch.from_numpy((np.asarray(ds.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); YTR = np.asarray(ds.targets)
XTE = torch.from_numpy((np.asarray(dst.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); YTE = np.asarray(dst.targets)
VFLIP = lambda x: x.flip(-2); SWAP = lambda x: x[:, [2, 1, 0]]          # orthogonal pixel maps that commute with isotropic noise
SIG = np.geomspace(20.0, 0.08, 56)                                       # descending noise grid (~10.5% steps)
print('noise grid', SIG[0], '...', SIG[-1])

# ---------------- classifiers ----------------
class FiLMBlock(nn.Module):
    def __init__(self, cin, cout, emb, stride=1):
        super().__init__(); self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout); self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.film = nn.Linear(emb, 2*cout); self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout)) if (stride != 1 or cin != cout) else nn.Identity()
    def forward(self, x, e):
        g, b = self.film(e).chunk(2, 1); h = F.silu(self.b1(self.c1(x))); h = self.b2(self.c2(h))*(1+g[:, :, None, None]) + b[:, :, None, None]; return F.silu(h + self.sc(x))
class NoisyResNet(nn.Module):
    """The paper's classifier (ResNet with FiLM noise conditioning)."""
    def __init__(self, n_out=10, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(32, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(3, ch, 3, 1, 1)
        self.blocks = nn.ModuleList([FiLMBlock(ch, ch, emb), FiLMBlock(ch, 2*ch, emb, 2), FiLMBlock(2*ch, 2*ch, emb), FiLMBlock(2*ch, 4*ch, emb, 2), FiLMBlock(4*ch, 4*ch, emb)]); self.fc = nn.Linear(4*ch, n_out)
    def forward(self, x, sigma):
        f = torch.exp(torch.arange(16, device=x.device)*-(math.log(100)/15)); t = 0.25*torch.log(sigma)[:, None]*f; e = self.emb(torch.cat([torch.sin(t), torch.cos(t)], 1)); h = self.inp(x)
        for b in self.blocks: h = b(h, e)
        return self.fc(h.mean((2, 3)))
class NoisyVGG(nn.Module):
    """A deliberately different family: plain VGG stack (no residuals), GroupNorm, input preconditioning x/sqrt(1+sigma^2),
    noise level entering as an extra constant input channel (no FiLM)."""
    def __init__(self, n_out=10, w=64):
        super().__init__(); L = []; cin = 4
        for cout, pool in [(w, False), (w, True), (2*w, False), (2*w, True), (4*w, False), (4*w, True)]:
            L += [nn.Conv2d(cin, cout, 3, 1, 1), nn.GroupNorm(8, cout), nn.GELU()] + ([nn.MaxPool2d(2)] if pool else []); cin = cout
        self.body = nn.Sequential(*L); self.fc = nn.Linear(cin, n_out)
    def forward(self, x, sigma):
        s = sigma.view(-1, 1, 1, 1); x = x/torch.sqrt(1 + s**2); c = (0.25*torch.log(s)).expand(-1, 1, x.shape[2], x.shape[3])
        return self.fc(self.body(torch.cat([x, c], 1)).mean((2, 3)))
def train_clf(model, X, Y, seed, epochs=25, bs=256, s_lo=0.05, s_hi=10.0, tag=''):
    torch.manual_seed(seed); Xd = X.to(dev); Yd = torch.from_numpy(Y).long().to(dev); opt = torch.optim.AdamW(model.parameters(), 2e-3, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=epochs*(len(Xd)//bs)); t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xd), device=dev)
        for i in range(0, len(Xd)-bs+1, bs):
            j = perm[i:i+bs]; x0 = Xd[j]
            if torch.rand(1).item() < 0.5: x0 = x0.flip(-1)                       # horizontal flip commutes with VFLIP and SWAP
            sigma = torch.exp(torch.empty(bs, device=dev).uniform_(math.log(s_lo), math.log(s_hi))); xt = x0 + sigma.view(-1, 1, 1, 1)*torch.randn_like(x0)
            with torch.autocast('cuda', dtype=torch.float16, enabled=(dev == 'cuda')): loss = F.cross_entropy(model(xt, sigma), Yd[j])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    model.eval(); del Xd; torch.cuda.empty_cache(); print(f"trained {tag} ({time.time()-t0:.0f}s)"); return model
@torch.no_grad()
def logits_grid(model, X, repeats, seed, transform=None, bs=1000):
    """Logits of model at every noise level of SIG for X (optionally transformed AFTER noising), R noise draws per image. Returns float16 [len(SIG), R*len(X), C]."""
    g = torch.Generator(device=dev).manual_seed(seed); C_ = None; out = None
    for r in range(repeats):
        for i0 in range(0, len(X), bs):
            x0 = X[i0:i0+bs].to(dev); eps = torch.randn(x0.shape, generator=g, device=dev)
            for gi, s in enumerate(SIG):
                xt = x0 + float(s)*eps
                if transform is not None: xt = transform(xt)
                with torch.autocast('cuda', dtype=torch.float16, enabled=(dev == 'cuda')): l = model(xt, torch.full((len(xt),), float(s), device=dev)).float()
                if out is None: C_ = l.shape[1]; out = np.zeros((len(SIG), repeats*len(X), C_), np.float16)
                out[gi, r*len(X)+i0: r*len(X)+i0+len(x0)] = l.cpu().numpy().astype(np.float16)
    return out

In [ ]:
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def kde_score(m, bwf=1.5):
    m = np.clip(m, -8, 8); bw = max(bwf*1.06*m.std()*len(m)**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, merge=0.03, prominence=0.05, lo=-3.5, hi=3.5):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx], merge)
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp_fit(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def standardize(z, ya):
    mua, mub = z[ya].mean(), z[~ya].mean(); half = 0.5*(mua-mub); sep = abs(half)/(0.5*(z[ya].std()+z[~ya].std())); return (z - 0.5*(mua+mub))/half, sep
def census_from_z(Zg, ya, bwf=1.5, merge=0.03, prominence=0.05, persist=1, min_sep=0.3):
    """Zg: list over the (descending) noise grid of raw coordinate values for the pair's samples; ya: boolean 'is class a'.
    Returns dict(sigma, gi, h, a, b, clamped) at the first bimodal grid point (bimodality must persist for `persist` consecutive points)."""
    seen_uni = False; run = 0
    for gi in range(len(SIG)):
        m, sep = standardize(Zg[gi], ya)
        if sep < min_sep: seen_uni = True; run = 0; continue
        sm, p = kde_score(m, bwf); nm = n_max(stationary(sm, p, merge, prominence))
        if nm >= 2 and seen_uni:
            run += 1
            if run >= persist:
                g0 = gi - persist + 1; m0, _ = standardize(Zg[g0], ya); sm0, p0 = kde_score(m0, bwf); return dict(sigma=float(SIG[g0]), gi=g0, **cusp_fit(sm0, p0))
        elif nm == 1: seen_uni = True; run = 0
    return None
def pair_rows(Y, a, b): return np.where((Y == a) | (Y == b))[0]
def z_pair(L, rows, ca, cb): return [L[gi, rows, ca].astype(np.float64) - L[gi, rows, cb].astype(np.float64) for gi in range(len(SIG))]
def eq_imbalance(h): hh = np.clip(np.abs(h), 0, 0.95); r = np.exp(2*np.arctanh(hh)); return r/(1+r)      # h ~ tanh(eta) near the transition; weight share of the favoured class
print('ok')

In [ ]:
# ---- load Step 20 outputs; recompute the null-pair logits with the saved 30-class classifier ----
P = glob.glob('/kaggle/input/**/step20_logits.npz', recursive=True); assert P, 'mount the Step 20 output'
Z20 = np.load(P[0]); L_MAIN, L_VGG = Z20['L_MAIN'], Z20['L_VGG']; assert np.allclose(Z20['SIG'], SIG)
YT4 = np.tile(YTE, 4); YT2 = np.tile(YTE, 2); IMG4 = np.tile(np.arange(len(YTE)), 4); IMG2 = np.tile(np.arange(len(YTE)), 2)
CLF30 = NoisyResNet(30).to(dev); CLF30.load_state_dict(torch.load(glob.glob('/kaggle/input/**/noisy_resnet30.pt', recursive=True)[0], map_location=dev)); CLF30.eval()
t0 = time.time()
L30_O = logits_grid(CLF30, XTE, 2, 21); L30_V = logits_grid(CLF30, VFLIP(XTE), 2, 22); L30_S = logits_grid(CLF30, SWAP(XTE), 2, 23)
print(f'null logits recomputed ({time.time()-t0:.0f}s)')

# ---- monotone calibration with cross-fitting ----
from scipy.optimize import minimize
def _basis(zs, knots, s):
    """Increasing basis: z, convex ramps s*softplus((z-t)/s) and concave ramps -s*softplus(-(z-t)/s). Any positive combination is increasing."""
    return np.column_stack([zs] + [s*np.logaddexp(0.0, (zs - t)/s) for t in knots] + [-s*np.logaddexp(0.0, -(zs - t)/s) for t in knots])
from scipy.special import expit
def mono_fit(z, y, n_knots=7, lam=1e-6):
    """Maximum-likelihood monotone calibration: u = b0 + B w with w >= 0 (bound-constrained), light ridge on the ramp weights."""
    mu, sd = z.mean(), z.std() + 1e-9; zs = (z - mu)/sd; knots = np.quantile(zs, np.linspace(0.03, 0.97, n_knots)); s = max(np.median(np.diff(knots)), 0.05)
    B = _basis(zs, knots, s); sgn = np.where(y, 1.0, -1.0); n = len(z); K = B.shape[1]
    def f(par):
        b0, w = par[0], par[1:]; u = b0 + B @ w; m = -sgn*u
        loss = np.mean(np.logaddexp(0.0, m)) + lam*np.sum(w[1:]**2); gu = -sgn*expit(m)/n
        gw = B.T @ gu; gw[1:] += 2*lam*w[1:]; return loss, np.concatenate([[gu.sum()], gw])
    par0 = np.concatenate([[0.0], [0.5], np.zeros(K-1)]); bounds = [(None, None)] + [(0.0, None)]*K
    r = minimize(f, par0, jac=True, method='L-BFGS-B', bounds=bounds); b0, w = r.x[0], r.x[1:]
    return lambda zz: b0 + _basis((zz - mu)/sd, knots, s) @ w
def crossfit(Zg, y, img):
    """Per noise level: calibrate on images with even id, score odd ones, and vice versa. Returns calibrated coordinate list and diagnostics."""
    A = (img % 2 == 0); out, diag = [], []
    for z in Zg:
        u = np.empty_like(z); fA = mono_fit(z[A], y[A]); fB = mono_fit(z[~A], y[~A]); u[~A] = fA(z[~A]); u[A] = fB(z[A]); out.append(u)
        lin = np.polyfit(z, u, 1); nl = np.std(u - np.polyval(lin, z))/max(np.std(u), 1e-9); diag.append((lin[0], nl))
    return out, diag
def tilt_check(u, y, nb=12):
    """Tilting identity: for a Bayes log-ratio, log p(u|a) - log p(u|b) = u (slope 1, intercept 0 for balanced pairs). Binned estimate on central 90%."""
    q = np.quantile(u, np.linspace(0.05, 0.95, nb+1)); c = 0.5*(q[1:]+q[:-1]); lr = []; w = []
    for lo, hi in zip(q[:-1], q[1:]):
        j = (u >= lo) & (u < hi); na, nb_ = (j & y).sum(), (j & ~y).sum(); lr.append(np.log((na+0.5)/(nb_+0.5)) - np.log((y.sum()+0.5)/((~y).sum()+0.5))); w.append(j.sum())
    sl, ic = np.polyfit(c, lr, 1, w=np.sqrt(w)); return sl, ic

# ---- validation of the calibration on known warps of an exact Bayes log-ratio (reported in the paper) ----
_r = np.random.default_rng(0); _n = 8000; _y = _r.random(_n) < 0.5; _zs = 2*(np.where(_y, 1.0, -1.0) + _r.standard_normal(_n))
for _nm, _w in [('affine', lambda s: 0.7*s + 0.3), ('tanh', lambda s: 3*np.tanh(s/3)), ('signed sqrt', lambda s: np.sign(s)*np.sqrt(np.abs(s))), ('cubic', lambda s: s + 0.08*s**3)]:
    _z = _w(_zs); _U, _d = crossfit([_z], _y, np.arange(_n)); _u = _U[0]; _c = np.abs(_zs) < np.percentile(np.abs(_zs), 95)
    print(f"validation {_nm:>11}: tilting slope raw {tilt_check(_z, _y)[0]:.2f} -> calibrated {tilt_check(_u, _y)[0]:.2f}; corr with true log-ratio {np.corrcoef(_u[_c], _zs[_c])[0,1]:.4f}; relative RMS error {np.std((_u-_zs)[_c])/np.std(_zs[_c]):.3f}; map nonlinearity {_d[0][1]:.3f}")
print('calibration machinery ready')

In [ ]:
# ---- calibrated census: ResNet and VGG, real pairs; 30-class real and null pairs ----
def calibrated_pair(L, Y_rows_labels, rows, ca, cb, img):
    Zg = z_pair(L, rows, ca, cb); y = Y_rows_labels
    Ug, diag = crossfit(Zg, y, img); return Zg, Ug, diag
t0 = time.time(); RES = {}; CAL = {}; VGR = {}; VGC = {}; DIAG = {}; TILT = {}
for a, b in PAIRS:
    rows = pair_rows(YT4, a, b); y = YT4[rows] == a; img = IMG4[rows]
    Zg, Ug, diag = calibrated_pair(L_MAIN, y, rows, a, b, img); RES[(a, b)] = census_from_z(Zg, y); CAL[(a, b)] = census_from_z(Ug, y); DIAG[(a, b)] = diag
    gi = CAL[(a, b)]['gi'] if CAL[(a, b)] else len(SIG)//2; TILT[(a, b)] = (tilt_check(Zg[gi], y), tilt_check(Ug[gi], y))
    Zv, Uv, _ = calibrated_pair(L_VGG, y, rows, a, b, img); VGR[(a, b)] = census_from_z(Zv, y); VGC[(a, b)] = census_from_z(Uv, y)
print(f'calibrated censuses ({time.time()-t0:.0f}s)')
tr = np.array([TILT[p][0] for p in PAIRS]); tc = np.array([TILT[p][1] for p in PAIRS])
print(f"tilting identity at each pair's transition (slope should be 1, intercept 0):  raw log-odds slope median {np.median(tr[:,0]):.2f} (IQR {np.percentile(tr[:,0],25):.2f}-{np.percentile(tr[:,0],75):.2f});  calibrated slope median {np.median(tc[:,0]):.2f} (IQR {np.percentile(tc[:,0],25):.2f}-{np.percentile(tc[:,0],75):.2f}), intercept median {np.median(np.abs(tc[:,1])):.2f}")
nlr = np.array([DIAG[p][CAL[p]['gi']][1] if CAL[p] else np.nan for p in PAIRS]); print(f"nonlinearity of the monotone calibration map at the transition: median {np.nanmedian(nlr):.3f}, max {np.nanmax(nlr):.3f}")
def agree(A, B, lab):
    ok = [p for p in PAIRS if A[p] and B[p]]; ha = np.array([A[p]['h'] for p in ok]); hb = np.array([B[p]['h'] for p in ok]); sa = np.array([A[p]['sigma'] for p in ok]); sb = np.array([B[p]['sigma'] for p in ok])
    print(f"{lab:>44}: n={len(ok)}, corr h {np.corrcoef(ha, hb)[0,1]:.3f}, sign agree {np.mean(np.sign(ha)==np.sign(hb)):.2f}, median |dh| {np.median(np.abs(ha-hb)):.3f}; corr log sigma {np.corrcoef(np.log(sa), np.log(sb))[0,1]:.3f}, median ratio {np.median(sb/sa):.2f}")
agree(RES, CAL, 'ResNet raw vs ResNet calibrated'); agree(VGR, VGC, 'VGG raw vs VGG calibrated')
agree(RES, VGR, 'ResNet raw vs VGG raw'); agree(CAL, VGC, 'ResNet calibrated vs VGG calibrated')

# null along the calibrated coordinate (30-class classifier), and real pairs through the same classifier
def null_cal(L_orig, L_trans, off):
    out = []
    for c in range(10):
        ra = np.where(YT2 == c)[0]; y = np.concatenate([np.ones(len(ra), bool), np.zeros(len(ra), bool)]); img = np.concatenate([IMG2[ra], IMG2[ra]])
        Zg = [np.concatenate([L_orig[gi, ra, c].astype(np.float64) - L_orig[gi, ra, c+off], L_trans[gi, ra, c].astype(np.float64) - L_trans[gi, ra, c+off]]) for gi in range(len(SIG))]
        Ug, _ = crossfit(Zg, y, img); out.append((census_from_z(Zg, y), census_from_z(Ug, y)))
    return out
NV = null_cal(L30_O, L30_V, 10); NS = null_cal(L30_O, L30_S, 20)
null_raw = np.array([abs(r[0]['h']) for r in NV+NS if r[0]]); null_c = np.array([abs(r[1]['h']) for r in NV+NS if r[1]])
R30r, R30c = [], []
for a, b in PAIRS:
    rows = pair_rows(YT2, a, b); y = YT2[rows] == a; Zg = z_pair(L30_O, rows, a, b); Ug, _ = crossfit(Zg, y, IMG2[rows]); R30r.append(census_from_z(Zg, y)); R30c.append(census_from_z(Ug, y))
h30r = np.array([abs(r['h']) for r in R30r if r]); h30c = np.array([abs(r['h']) for r in R30c if r])
print(f"null |h| (symmetric pairs through the classifier): raw median {np.median(null_raw):.4f}, max {null_raw.max():.4f} (n={len(null_raw)});  calibrated median {np.median(null_c):.4f}, max {null_c.max():.4f} (n={len(null_c)})")
print(f"real pairs through the same classifier: raw |h| median {np.median(h30r):.3f}, above raw-null max {np.sum(h30r > null_raw.max())}/{len(h30r)};  calibrated |h| median {np.median(h30c):.3f}, above calibrated-null max {np.sum(h30c > null_c.max())}/{len(h30c)}")

In [ ]:
# ---- bootstrap (300) along the calibrated ResNet coordinate; effect sizes; class-level tests; hierarchy; Gaussian baseline ----
NB = 300; BC = {}; t0 = time.time(); UG = {}
for a, b in PAIRS:
    rows = pair_rows(YT4, a, b); y = YT4[rows] == a; Ug, _ = crossfit(z_pair(L_MAIN, rows, a, b), y, IMG4[rows]); UG[(a, b)] = (Ug, y)
    if not CAL[(a, b)]: continue
    ia = np.where(y)[0]; ib = np.where(~y)[0]; hs, ss = [], []
    for _ in range(NB):
        sel = np.concatenate([rng.choice(ia, len(ia)), rng.choice(ib, len(ib))]); r = census_from_z([u[sel] for u in Ug], y[sel])
        if r: hs.append(r['h']); ss.append(r['sigma'])
    BC[(a, b)] = (np.array(hs), np.array(ss))
print(f'bootstrap ({time.time()-t0:.0f}s)')
NULLMAX = null_c.max(); okp = [p for p in PAIRS if p in BC]
print(f"{'pair':>22} {'sigma_bif [95% CI]':>22} {'h [95% CI]':>26} {'equiv. split':>12} {'beyond null':>12}")
Hc, sig_med = [], {}
for p in okp:
    hs, ss = BC[p]; hmed = np.median(hs); hlo, hhi = np.percentile(hs, [2.5, 97.5]); smed = np.median(ss); slo, shi = np.percentile(ss, [2.5, 97.5]); sig_med[p] = smed; Hc.append(hmed)
    beyond = (abs(hmed) > NULLMAX) and (hlo > 0 or hhi < 0)
    print(f"{CLASSES[p[0]]+'/'+CLASSES[p[1]]:>22} {smed:7.2f} [{slo:5.2f},{shi:5.2f}] {hmed:+8.3f} [{hlo:+.3f},{hhi:+.3f}] {100*eq_imbalance(hmed):8.0f}/{100-100*eq_imbalance(hmed):.0f} {str(beyond):>12}")
Hc = np.array(Hc); print(f"calibrated coordinate: median |h| {np.median(np.abs(Hc)):.3f} (equivalent split {100*eq_imbalance(np.median(np.abs(Hc))):.0f}/{100-100*eq_imbalance(np.median(np.abs(Hc))):.0f}); "
      f"beyond the calibrated through-classifier null: {sum((abs(np.median(BC[p][0])) > NULLMAX) and (np.percentile(BC[p][0],2.5) > 0 or np.percentile(BC[p][0],97.5) < 0) for p in okp)}/{len(okp)}")
VAR = np.array([float(XTR[YTR == c].var(0).mean()) for c in range(10)])
def rule(var): dv = np.array([var[a]-var[b] for a, b in okp]); return np.corrcoef(Hc, dv)[0, 1], int(np.sum(np.sign(Hc) == -np.sign(dv)))
r_obs, n_obs = rule(VAR); pr = np.array([rule(rng.permutation(VAR)) for _ in range(10000)])
print(f"tighter-class rule (calibrated): r = {r_obs:.3f} (class-permutation p = {np.mean(pr[:,0] <= r_obs):.4f}); count {n_obs}/{len(okp)} (p = {np.mean(pr[:,1] >= n_obs):.4f})")
from itertools import combinations as comb
def hier(veh):
    kd = lambda p: 'va' if ((p[0] in veh) != (p[1] in veh)) else ('vv' if p[0] in veh else 'aa')
    return np.median([sig_med[p] for p in okp if kd(p) == 'va'])/np.median([sig_med[p] for p in okp if kd(p) == 'aa'])
obs = hier(VEH); allr = np.array([hier(set(v)) for v in comb(range(10), 4)])
print(f"hierarchy (calibrated): va/aa median ratio {obs:.2f}; rank {int(np.sum(allr >= obs))}/210 (p = {np.mean(allr >= obs):.4f})")
kind = lambda p: 'va' if ((p[0] in VEH) != (p[1] in VEH)) else ('vv' if p[0] in VEH else 'aa')
for k in ['va', 'vv', 'aa']: print(f"   {k}: median calibrated sigma_bif {np.median([sig_med[p] for p in okp if kind(p) == k]):.2f}")
Xf = XTR.reshape(len(XTR), -1).to(dev).double(); SG = {}
for a, b in okp:
    A_ = Xf[torch.from_numpy(YTR == a).to(dev)]; B_ = Xf[torch.from_numpy(YTR == b).to(dev)]; dmu = A_.mean(0) - B_.mean(0)
    lam, E = torch.linalg.eigh(0.5*(torch.cov(A_.T) + torch.cov(B_.T))); prj = ((E.T @ dmu)**2).cpu().numpy(); lam = lam.clamp_min(0).cpu().numpy()
    dp = lambda s: np.sqrt(np.sum(prj/(lam + s**2))); lo, hi = 1e-3, 100.0
    if dp(lo) < 2: SG[(a, b)] = np.nan; continue
    for _ in range(60): mid = np.sqrt(lo*hi); lo, hi = (mid, hi) if dp(mid) > 2 else (lo, mid)
    SG[(a, b)] = np.sqrt(lo*hi)
del Xf; torch.cuda.empty_cache()
okg = [p for p in okp if np.isfinite(SG[p])]; sg = np.log([SG[p] for p in okg]); sb = np.log([sig_med[p] for p in okg]); res = sb - np.polyval(np.polyfit(sg, sb, 1), sg)
print(f"Gaussian baseline vs calibrated sigma_bif: corr(log) {np.corrcoef(sg, sb)[0,1]:.3f}; residual by category: " + ', '.join(f"{k} {np.median(res[[i for i,p in enumerate(okg) if kind(p)==k]]):+.2f}" for k in ['va','vv','aa']))
np.savez_compressed('/kaggle/working/step20b_calibrated.npz', pairs=np.array(okp), H=Hc, S=np.array([sig_med[p] for p in okp]),
                    HLO=np.array([np.percentile(BC[p][0], 2.5) for p in okp]), HHI=np.array([np.percentile(BC[p][0], 97.5) for p in okp]), null_c=null_c, null_raw=null_raw)

## What to report back
- **Calibration quality:** the tilting line (raw vs calibrated slope; the calibrated slope should be about 1) and the nonlinearity line.
- **Coordinate agreement:** the four agreement lines. The key comparison is whether ResNet and VGG agree at least as well after calibration as before.
- **Null and real pairs:** the null line and the real-pair line.
- **Calibrated census:** the per-pair table, the effect-size line, the tighter-class rule, the hierarchy and the Gaussian-baseline lines.